# Forecast Benchmark Dashboard

This notebook explores the short-horizon forecasting benchmark for Albanian urban air quality.


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Markdown


In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BENCHMARK_DIR = PROJECT_ROOT / "outputs" / "forecast_benchmark" / "tables"

daily = pd.read_csv(PROCESSED_DIR / "albania_air_quality_daily.csv", parse_dates=["date"])
summary = pd.read_csv(BENCHMARK_DIR / "model_summary.csv")
best = pd.read_csv(BENCHMARK_DIR / "best_models_by_city.csv")
latest = pd.read_csv(BENCHMARK_DIR / "latest_forecasts.csv", parse_dates=["latest_observed_date", "forecast_date"])

display(Markdown(
    f"Loaded benchmark outputs for **{summary['target'].nunique()}** targets, **{best['city'].nunique()}** cities, and **{summary['model'].nunique()}** forecasting models."
))
summary.head()


Loaded benchmark outputs for **2** targets, **8** cities, and **5** forecasting models.

,target,target_label,model,horizon_days,cities,mean_mae,mean_rmse,mean_mape,mean_smape,mean_rank
0,european_aqi_max,Daily Max European AQI,ARIMA,1,8,3.626567,4.919951,9.472221,9.269416,2.000
1,european_aqi_max,Daily Max European AQI,Naive,1,8,3.762681,5.294297,9.835046,9.608780,2.625
2,european_aqi_max,Daily Max European AQI,Random Forest Lag7,1,8,3.770311,5.011644,9.950590,9.667278,3.000
3,european_aqi_max,Daily Max European AQI,Drift,1,8,3.768326,5.300502,9.848895,9.622322,3.625
4,european_aqi_max,Daily Max European AQI,Holt Damped,1,8,3.827681,5.282473,10.046692,9.809302,3.750


In [3]:
target_lookup = {
    "Daily Mean PM2.5": "pm2_5_mean",
    "Daily Max European AQI": "european_aqi_max",
}

daily_target_map = {
    "pm2_5_mean": "pm2_5_mean",
    "european_aqi_max": "european_aqi_max",
}

target_widget = widgets.Dropdown(
    options=list(target_lookup.keys()),
    value="Daily Mean PM2.5",
    description="Target:",
    layout=widgets.Layout(width="320px"),
)

horizon_widget = widgets.ToggleButtons(
    options=[("1 day", 1), ("2 days", 2), ("3 days", 3)],
    value=1,
    description="Horizon:",
)

city_widget = widgets.Dropdown(
    options=sorted(daily["city"].unique()),
    value="Tirane",
    description="City:",
    layout=widgets.Layout(width="220px"),
)

history_widget = widgets.IntSlider(
    value=45,
    min=30,
    max=120,
    step=5,
    description="History:",
    continuous_update=False,
)

output = widgets.Output()


In [4]:
def render_dashboard(*_):
    output.clear_output(wait=True)

    target_label = target_widget.value
    target = target_lookup[target_label]
    horizon = horizon_widget.value
    city = city_widget.value
    history_days = history_widget.value

    summary_subset = summary[(summary['target'] == target) & (summary['horizon_days'] == horizon)].copy()
    summary_subset = summary_subset.sort_values(['mean_rank', 'mean_mae'])
    best_global = summary_subset.iloc[0]

    best_subset = best[(best['target'] == target) & (best['horizon_days'] == horizon)].copy()
    best_subset = best_subset.sort_values(['rank_mae', 'mae', 'city'])
    best_city_row = best_subset[best_subset['city'] == city].iloc[0]

    latest_subset = latest[(latest['target'] == target) & (latest['city'] == city)].copy()
    latest_subset = latest_subset.sort_values(['model', 'horizon_days'])

    daily_col = daily_target_map[target]
    history = daily[daily['city'] == city][['date', daily_col]].dropna().copy().sort_values('date').tail(history_days)

    summary_fig = px.bar(
        summary_subset,
        x='model',
        y='mean_mae',
        color='mean_rank',
        template='plotly_white',
        title=f'Model comparison - {target_label}, horizon {horizon} day(s)',
        labels={'mean_mae': 'Mean MAE', 'mean_rank': 'Mean rank'},
    )
    summary_fig.update_layout(height=420, xaxis_title='Model', yaxis_title='Mean MAE')

    forecast_fig = go.Figure()
    forecast_fig.add_trace(
        go.Scatter(
            x=history['date'],
            y=history[daily_col],
            mode='lines',
            name='Observed',
            line=dict(color='#1f77b4', width=3),
        )
    )
    for model_name, model_df in latest_subset.groupby('model'):
        forecast_fig.add_trace(
            go.Scatter(
                x=model_df['forecast_date'],
                y=model_df['predicted'],
                mode='lines+markers',
                name=model_name,
            )
        )
    forecast_fig.update_layout(
        template='plotly_white',
        height=460,
        title=f'Latest forecast paths - {city} / {target_label}',
        xaxis_title='Date',
        yaxis_title=target_label,
    )

    latest_pivot = latest_subset.pivot(index='model', columns='horizon_days', values='predicted').reset_index()
    latest_pivot = latest_pivot.rename(columns={1: 'day_1', 2: 'day_2', 3: 'day_3'})

    with output:
        display(Markdown(
            f"## Forecast Benchmark Snapshot\n"
            f"- Best overall model for **{target_label}** at **{horizon}-day** horizon: **{best_global['model']}**\n"
            f"- Mean MAE: **{best_global['mean_mae']:.3f}**\n"
            f"- Mean MAPE: **{best_global['mean_mape']:.2f}%**\n"
            f"- Best model for **{city}** at this horizon: **{best_city_row['model']}**"
        ))
        display(summary_fig)
        display(best_subset[['city', 'model', 'mae', 'rmse', 'mape', 'rank_mae']].reset_index(drop=True))
        display(forecast_fig)
        display(Markdown(f"### Latest forecast table - {city}"))
        display(latest_pivot)

for widget in [target_widget, horizon_widget, city_widget, history_widget]:
    widget.observe(render_dashboard, names='value')

display(widgets.HBox([target_widget, horizon_widget, city_widget, history_widget]))
display(output)
render_dashboard()


Output()

In [5]:
summary.sort_values(['target', 'horizon_days', 'mean_rank', 'mean_mae']).reset_index(drop=True)


,target,target_label,model,horizon_days,cities,mean_mae,mean_rmse,mean_mape,mean_smape,mean_rank
0,european_aqi_max,Daily Max European AQI,ARIMA,1,8,3.626567,4.919951,9.472221,9.269416,2.000
1,european_aqi_max,Daily Max European AQI,Naive,1,8,3.762681,5.294297,9.835046,9.608780,2.625
2,european_aqi_max,Daily Max European AQI,Random Forest Lag7,1,8,3.770311,5.011644,9.950590,9.667278,3.000
3,european_aqi_max,Daily Max European AQI,Drift,1,8,3.768326,5.300502,9.848895,9.622322,3.625
4,european_aqi_max,Daily Max European AQI,Holt Damped,1,8,3.827681,5.282473,10.046692,9.809302,3.750
5,european_aqi_max,Daily Max European AQI,ARIMA,2,8,4.582405,6.266595,12.152511,11.779290,1.500
6,european_aqi_max,Daily Max European AQI,Holt Damped,2,8,4.693294,6.491893,12.594813,12.008515,2.500
7,european_aqi_max,Daily Max European AQI,Naive,2,8,4.751812,6.569583,12.729583,12.137450,2.875
8,european_aqi_max,Daily Max European AQI,Drift,2,8,4.760085,6.579702,12.749218,12.157139,4.000
9,european_aqi_max,Daily Max European AQI,Random Forest Lag7,2,8,4.848529,6.517206,13.006154,12.372227,4.125
